# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets with their @id
record_sets = dataset.metadata.record_sets
print('Record Sets:')
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name', 'No name')}\n  Description: {rs.get('description', 'No description')}")

# For each record set, list fields and columns by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('fields', [])
    print('  Fields:')
    for f in fields:
        print(f"    - {f['@id']} : {f.get('name', 'No name')}\n      Description: {f.get('description', 'No description')}")
    columns = rs.get('columns', [])
    print('  Columns:')
    for c in columns:
        print(f"    - {c['@id']} : {c.get('name', 'No name')}\n      Description: {c.get('description', 'No description')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect record set IDs for extraction
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in '{record_set_id}': {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, and grouping by attributes.

In [ ]:
# Example: Assume main record set is the survey responses
# Replace with actual @id from your dataset's record sets overview
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    # Try to pick the main survey table
    if 'survey' in rs.get('name','').lower() or 'responses' in rs.get('name','').lower():
        main_record_set_id = rs['@id']
        break
if not main_record_set_id:
    main_record_set_id = record_set_ids[0]

# Use the main dataframe
df = dataframes.get(main_record_set_id)
if df is not None:
    # Find available numeric columns
    numeric_fields = [col for col in df.columns if df[col].dtype != 'O']
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        # Attempt to group by demographic column
        group_fields = [col for col in df.columns if 'education' in col.lower() or 'gender' in col.lower() or 'age' in col.lower()]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No demographic field found for grouping.")
    else:
        print("No numeric fields found in main record set.")
else:
    print("Survey record set DataFrame not loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if df is not None and not df.empty:
    # Visualize numeric field distribution
    if numeric_fields:
        plt.figure(figsize=(6,4))
        sns.histplot(df[numeric_field_id], bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()

    # If grouped_df exists, visualize group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and explored the FAIR² Mental Health Survey dataset using the `mlcroissant` library.
- Reviewed the record sets and fields uniquely referenced by their `@id` values.
- Performed preliminary EDA, including filtering, normalizing, and grouping on available numeric fields (e.g., survey scores).
- Visualized distributions and relationships between mental health survey results and demographic attributes.
- This process demonstrates how Croissant schemas support reproducible, AI-ready data workflows in public health research.